In [1]:
import numpy as np
import pandas as pd
import xarray as xr
import dask
import matplotlib.pyplot as plt
import os

from siphon.catalog import TDSCatalog

In [2]:
# change working directory
os.chdir('D:/UofG/Arctic-VBD')
print(os.getcwd()) # verify change

D:\UofG\Arctic-VBD


# 1. Saving climate data
Here we will use a publicly available downscaled gridded data from PCIC. PCIC used observations from NRCAN gridded data from 1950 - 2005 to perform BCCAQv2 downscaling.

In [3]:
# acquire downscaled data from PAVICS. Code adapted from https://utcdw.physics.utoronto.ca/UTCDW_Guidebook/README.html
url_pavics = "https://pavics.ouranos.ca/twitcher/ows/proxy/thredds/catalog/datasets/"
url_downscaled = url_pavics + "simulations/bias_adjusted/cmip6/pcic/CanDCS-U6/catalog.xml"
cat_sds = TDSCatalog(url_downscaled)

# open the downscaled gcm dataset and load the data
opendap_urls = [cat_sds.datasets[i].access_urls["OPENDAP"] for i in range(len(cat_sds.datasets))]
datasets_canesm_sds = list(filter(lambda x: 'CanESM' in x, opendap_urls))

datasets_canesm_sds

['https://pavics.ouranos.ca/twitcher/ows/proxy/thredds/dodsC/datasets/simulations/bias_adjusted/cmip6/pcic/CanDCS-U6/day_BCCAQv2+ANUSPLIN300_CanESM5_historical+ssp126_r10i1p2f1_gn_1950-2100.ncml',
 'https://pavics.ouranos.ca/twitcher/ows/proxy/thredds/dodsC/datasets/simulations/bias_adjusted/cmip6/pcic/CanDCS-U6/day_BCCAQv2+ANUSPLIN300_CanESM5_historical+ssp126_r1i1p2f1_gn_1950-2100.ncml',
 'https://pavics.ouranos.ca/twitcher/ows/proxy/thredds/dodsC/datasets/simulations/bias_adjusted/cmip6/pcic/CanDCS-U6/day_BCCAQv2+ANUSPLIN300_CanESM5_historical+ssp126_r2i1p2f1_gn_1950-2100.ncml',
 'https://pavics.ouranos.ca/twitcher/ows/proxy/thredds/dodsC/datasets/simulations/bias_adjusted/cmip6/pcic/CanDCS-U6/day_BCCAQv2+ANUSPLIN300_CanESM5_historical+ssp126_r3i1p2f1_gn_1950-2100.ncml',
 'https://pavics.ouranos.ca/twitcher/ows/proxy/thredds/dodsC/datasets/simulations/bias_adjusted/cmip6/pcic/CanDCS-U6/day_BCCAQv2+ANUSPLIN300_CanESM5_historical+ssp126_r4i1p2f1_gn_1950-2100.ncml',
 'https://pavics.ou

This time we will use CanESM5 ensemble member r1i1p2f1

## 1a. SSP2-4.5

In [4]:
ssp245_url = datasets_canesm_sds[11] #SSP2-4.5; ensemble member: r1i1p2f1
print(ssp245_url)
ssp245 = xr.open_dataset(ssp245_url)[['tasmax','tasmin']]

https://pavics.ouranos.ca/twitcher/ows/proxy/thredds/dodsC/datasets/simulations/bias_adjusted/cmip6/pcic/CanDCS-U6/day_BCCAQv2+ANUSPLIN300_CanESM5_historical+ssp245_r1i1p2f1_gn_1950-2100.ncml


We need daily mean temperature. However, this dataset only contains the daily maximum and daily minimum temperatures. To get the daily mean temperature, we will take the average of the daily maximum and daily minimum. Since the dataset is too large, we will subset the dataset by year and compute the mean temperature.

In [5]:
start_year = 1950
end_year = 2100

# Loop through each year
for year in range(start_year, end_year + 1):

    path = "data-raw/climate-data/"
    outfile = path + f"ssp245/tas_day_CanESM5_ssp245_r1i1p2f1_{year}.nc"

    if not os.path.exists(outfile):
        print(f"Processing year {year}...")
        
        # select that year
        ds_year = ssp245.sel(time = ssp245.time.dt.year == year)

        # Calculate the mean temperature
        ds_year['tas'] = (ds_year['tasmin'] + ds_year['tasmax'])/2
        ds_year.tas.attrs['units'] = 'DegC'
    
        print(f"Finished processing year {year}. Now saving file...")
        # Save as NetCDF file
        ds_year.to_netcdf(outfile)
        print(f"Saved {outfile}")

    else:
        print(f"tas_day_CanESM5_ssp245_r1i1p2f1_{year}.nc already downloaded")

tas_day_CanESM5_ssp245_r1i1p2f1_1950.nc already downloaded
tas_day_CanESM5_ssp245_r1i1p2f1_1951.nc already downloaded
tas_day_CanESM5_ssp245_r1i1p2f1_1952.nc already downloaded
tas_day_CanESM5_ssp245_r1i1p2f1_1953.nc already downloaded
tas_day_CanESM5_ssp245_r1i1p2f1_1954.nc already downloaded
tas_day_CanESM5_ssp245_r1i1p2f1_1955.nc already downloaded
tas_day_CanESM5_ssp245_r1i1p2f1_1956.nc already downloaded
tas_day_CanESM5_ssp245_r1i1p2f1_1957.nc already downloaded
tas_day_CanESM5_ssp245_r1i1p2f1_1958.nc already downloaded
tas_day_CanESM5_ssp245_r1i1p2f1_1959.nc already downloaded
tas_day_CanESM5_ssp245_r1i1p2f1_1960.nc already downloaded
tas_day_CanESM5_ssp245_r1i1p2f1_1961.nc already downloaded
tas_day_CanESM5_ssp245_r1i1p2f1_1962.nc already downloaded
tas_day_CanESM5_ssp245_r1i1p2f1_1963.nc already downloaded
tas_day_CanESM5_ssp245_r1i1p2f1_1964.nc already downloaded
tas_day_CanESM5_ssp245_r1i1p2f1_1965.nc already downloaded
tas_day_CanESM5_ssp245_r1i1p2f1_1966.nc already download

## 1b. SSP1-2.6

In [7]:
ssp126_url = datasets_canesm_sds[1] #SSP1-2.6; ensemble member: r1i1p2f1
print(ssp126_url)
ssp126 = xr.open_dataset(ssp126_url)[['tasmax','tasmin']]

https://pavics.ouranos.ca/twitcher/ows/proxy/thredds/dodsC/datasets/simulations/bias_adjusted/cmip6/pcic/CanDCS-U6/day_BCCAQv2+ANUSPLIN300_CanESM5_historical+ssp126_r1i1p2f1_gn_1950-2100.ncml


In [ ]:
start_year = 1950
end_year = 2100

# Loop through each year
for year in range(start_year, end_year + 1):

    path = "data-raw/climate-data/"
    outfile = path + f"ssp126/tas_day_CanESM5_ssp126_r1i1p2f1_{year}.nc"

    if not os.path.exists(outfile):
        print(f"Processing year {year}...")
        
        # select that year
        ds_year = ssp126.sel(time = ssp126.time.dt.year == year)

        # Calculate the mean temperature
        ds_year['tas'] = (ds_year['tasmin'] + ds_year['tasmax'])/2
        ds_year.tas.attrs['units'] = 'DegC'
    
        print(f"Finished processing year {year}. Now saving file...")
        # Save as NetCDF file
        ds_year.to_netcdf(outfile)
        print(f"Saved {outfile}")

    else:
        print(f"tas_day_CanESM5_ssp126_r1i1p2f1_{year}.nc already downloaded")